# <a id='toc1_'></a>[Homework Assignment: Develop Your Own Color Jittering Animation](#toc0_)


**Table of contents**<a id='toc0_'></a>    
- [Homework Assignment: Develop Your Own Color Jittering Animation](#toc1_)    
  - [Your Goal](#toc1_1_)    
  - [Step-by-Step Requirements](#toc1_2_)    
    - [Set Up the Basics](#toc1_2_1_)    
    - [Write a Checkerboard Function](#toc1_2_2_)    
    - [Build the Animation](#toc1_2_3_)    
    - [Add a Color Reference Panel (10 points)](#toc1_2_4_)    
    - [Document Your Work](#toc1_2_5_)    
  - [Example Structure](#toc1_3_)    
  - [What to Submit](#toc1_4_)    
  - [Hints](#toc1_5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

Hello students! This week, you’ll build a Python animation that demonstrates **color jittering**—the fascinating effect where tiny color patterns blend into new hues. By the end, you’ll understand how digital screens create millions of colors from just a few base shades.  

## <a id='toc1_1_'></a>[Your Goal](#toc0_)
Create a Jupyter Notebook that generates an animation showing a checkerboard pattern of two colors. As the grid gets denser (from 2x2 to 256x256), the colors should optically blend into a new perceived color.  

## <a id='toc1_2_'></a>[Step-by-Step Requirements](#toc0_)

### <a id='toc1_2_1_'></a>[Set Up the Basics](#toc0_)
- Import required libraries: `numpy`, `matplotlib.pyplot`, `FuncAnimation` from `matplotlib.animation`, and `HTML` from `IPython.display`.  
- Define an image size that’s a power of 2 (e.g., 256x256 or 512x512 pixels) to ensure perfect grid alignment.  
- Choose two base colors (e.g., red and blue) and define their RGB values (use 0-1 range, like `[1,0,0]` for red).  

In [100]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import cv2

# 定义图像大小（必须是2的次方）
img_size = 256 

# 定义两种基色 (RGB, 范围0~1)
color1 = np.array([1, 1, 0])   # 黄色
color2 = np.array([1, 0.5, 0])   # 橙色


In [101]:
%matplotlib inline
plt.ioff()   # 关闭交互式绘制

### <a id='toc1_2_2_'></a>[Write a Checkerboard Function](#toc0_)
Create a function called `create_checkerboard` that:  
- Takes 3 inputs: `grid_size` (number of squares per side), `img_size` (image dimensions), and the two base colors.  
- Returns a `NumPy` array (the checkerboard image) where squares alternate between the two colors.  
- Ensures no "black edges"—each square must perfectly fill its space (hint: use integer division to calculate square size).  

In [102]:
def create_checkerboard(img_size=256, color1=(1, 0, 0), color2=(0, 0, 1), square_size=32):
    # 创建网格索引
    x = np.arange(img_size)
    y = np.arange(img_size)
    xx, yy = np.meshgrid(x, y)

    # 初始化图像
    img = np.zeros((img_size, img_size, 3))

    # 创建图形和轴
    fig, ax = plt.subplots()
    im = ax.imshow(img)
    ax.axis("off")

    
    # 创建移动的棋盘格模式
    checker = (xx // (256/square_size) + yy // (256/square_size)) % 2
    # 根据checker选择颜色
    img[checker == 0] = color1
    img[checker == 1] = color2
    im.set_array(img)


    return img


### <a id='toc1_2_3_'></a>[Build the Animation](#toc0_)
- Define a list of grid sizes that double each time (e.g., `[2, 4, 8, 16, 32, 64, 128, 256]`).  
- Use `FuncAnimation` to create an animation that:  
  - Starts with the largest grid (2x2) and progresses to the smallest (256x256).  
  - Updates the title to show the current grid size (e.g., "Grid: 32x32").  
  - Runs at 1 frame per second and loops continuously.  

In [103]:
grid_sizes = [2, 4, 8, 16, 32, 64, 128, 256]

# 创建画布
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
im = ax1.imshow(create_checkerboard(img_size, color1, color2, grid_sizes[0]))
title = ax1.set_title(f"Grid: {grid_sizes[0]}x{grid_sizes[0]}")

# 更新函数
def update(frame):
    g = grid_sizes[frame % len(grid_sizes)]  # 循环使用
    img = create_checkerboard(img_size, color1, color2, g)
    im.set_data(img)
    title.set_text(f"Grid: {g}x{g}")
    return [im, title]

# 创建动画 (1 fps = interval 1000ms)
ani = FuncAnimation(
    fig,
    update,
    frames=len(grid_sizes),
    interval=1000,
    blit=False,
    repeat=True
)


### <a id='toc1_2_4_'></a>[Add a Color Reference Panel](#toc0_)
Include a side-by-side panel that shows:  
- The two base colors.  
- The "predicted blended color" (what you think the final grid will look like).  

In [104]:
# 右图：颜色面板
panel = np.ones((img_size, img_size, 3))
ax2.imshow(panel)
ax2.axis("off")
ax2.set_title("Base Colors & Blended")

# 预测混合色
blended_color = (color1 + color2) / 2

# 在右侧面板中绘制三个矩形区域
rect_w = img_size // 3
panel[:, :rect_w, :] = color1          # 左侧 = 基色1
panel[:, rect_w:2*rect_w, :] = color2  # 中间 = 基色2
panel[:, 2*rect_w:, :] = blended_color # 右侧 = 混合色
panel_im = ax2.imshow(panel)


HTML(ani.to_jshtml())  # 在Jupyter中显示动画

### <a id='toc1_2_5_'></a>[Document Your Work](#toc0_)
Add comments explaining:  
- Why you chose your two base colors.  
- How the checkerboard function avoids black edges.  
- At what grid size the colors first appear blended to your eye.  

In [105]:
# I chose the colors yellow and orange because they look like the ice cream bars I like.
# To avoid black edges, I made sure the image size is a power of 2 (256x256) and the grid sizes divide evenly into the image size by using integer division.
# The colors appear to blend when the grid size is 128x128 or smaller.

## <a id='toc1_3_'></a>[Example Structure](#toc0_)
Your notebook should flow like this:  
1. Library imports  
2. Color and animation parameters (image size, grid sizes, base colors)  
3. `create_checkerboard` function  
4. Animation setup (figure, initial plot, update function)  
5. Display the animation with `HTML(animation.to_html5_video())`  
6. Written comments/analysis

## <a id='toc1_4_'></a>[What to Submit](#toc0_)
- Html of your Jupyter Notebook.

## <a id='toc1_5_'></a>[Hints](#toc0_)
- Use `np.repeat` to scale up small grids into full-sized images.  
- Test your `create_checkerboard` function with a 2x2 grid first—make sure it works before animating!  
- If you see black edges, check that your image size is divisible by all grid sizes.

This project lets you combine coding skills with color science—have fun, and don’t hesitate to ask for help!  

Due: 12pm, 1/Oct/2025
Happy coding!